# 1. CP8EP8, S = 131072.

## Assumptions

- Assume all GPUs have even routing (the tokens are evenly distributed to experts and thus each GPU receives the same amount of tokens)
- NVLink is at full efficiency; 900GB/s unidirectional.

## EXPECTATION:

S/CP = 16384 tokens per GPU.

On one GPU, we send 16384, and receive 16384.

topk (k=8) routing, so we send 16384 * 8 tok.

1/8 are on the source, 7/8 are on other GPUs. So send 16384 * 8 * 7/8 toks.

Each tok has h=6144, and is in bf16 (2 bytes)

So 16384 * 8 * 7/8 * 6144 * 2 bytes sent, over NVLink which is 900GB/s unidirectional.
= 1.566ms.

## REALITY:

We see on rank 0, 5.131ms.

Some assumptions must not hold. Which ones?

Correction: this is assuming even routing, but this assumption does not hold.

For rank r, the lower bound is roughly:

```text
T_r >= max(
    remote bytes sent by rank r / effective egress bandwidth,
    remote bytes received by rank r / effective ingress bandwidth
)
```

effective egress/ingress bandwidth is assumed to be 900GB/s, but can be lower if we don't achieve full NVLink efficiency.

The ncclDevKernel_SendRecv in the trace is T_r for that rank; this duration can differ across ranks. They must synchronize after, but the recording of this kernel can differ across different ranks.

On rank 0, we observe 5.13ms.

This is because, firstly, we have the trace so we can see how many tokens were sent/received on rank 0:

### For rank 0:

Total sent:      131072
Local self-send:  30279
Remote sent:     100793

Total received:  241130
Local self-recv:  30279
Remote received: 210851

send bound    = 100793 × 6144 × 2 / 900 GB/s = 1.376 ms
receive bound = 210851 × 6144 × 2 / 900 GB/s = 2.879 ms

T_rank0 >= max(1.376, 2.879) = 2.879 ms

However it could be limited even further by a hotter receiver - its ingress would be bottlenecked. So we should look at the profile across all ranks to see where the hotspot is.


I did another trace of the same thing but this time recorded the runtime profiles of all ranks, not just rank 0.  

`t_start` and `t_end` are in milliseconds, relative to the earliest token-dispatch `ncclDevKernel_SendRecv` start across all ranks.


| Rank | Received | Multiple of balanced | t_start (ms) | t_end (ms) | t_elapsed (ms) |
| ---- | -------- | -------------------- | ------------ | ---------- | -------------- |
| 0    | 241130   | 1.840×               | 0.000000     | 5.125728   | 5.125728       |
| 1    | 2772     | 0.021×               | 0.541533     | 4.671231   | 4.129698       |
| 2    | 247832   | 1.891×               | 1.020281     | 5.387824   | 4.367543       |
| 3    | 108328   | 0.826×               | 0.557426     | 5.186709   | 4.629283       |
| 4    | 3042     | 0.023×               | 0.266277     | 3.993148   | 3.726871       |
| 5    | 258648   | 1.973×               | 0.133229     | 4.919745   | 4.786516       |
| 6    | 20181    | 0.154×               | 0.134093     | 3.931563   | 3.797470       |
| 7    | 166643   | 1.271×               | 0.272354     | 5.393879   | 5.121525       |

| Source \ Destination | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | Total sent |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 0 | 30281 | 356 | 30692 | 13472 | 416 | 32459 | 2538 | 20858 | 131072 |
| 1 | 30120 | 371 | 30913 | 13572 | 416 | 32337 | 2453 | 20890 | 131072 |
| 2 | 30110 | 361 | 30946 | 13554 | 394 | 32367 | 2506 | 20834 | 131072 |
| 3 | 30041 | 331 | 31111 | 13543 | 379 | 32292 | 2552 | 20823 | 131072 |
| 4 | 30099 | 314 | 31151 | 13507 | 357 | 32239 | 2569 | 20836 | 131072 |
| 5 | 30040 | 356 | 31125 | 13572 | 369 | 32431 | 2449 | 20730 | 131072 |
| 6 | 30349 | 331 | 30995 | 13541 | 327 | 32149 | 2538 | 20842 | 131072 |
| 7 | 30090 | 352 | 30899 | 13567 | 384 | 32374 | 2576 | 20830 | 131072 |
| **Total received** | **241130** | **2772** | **247832** | **108328** | **3042** | **258648** | **20181** | **166643** | **1048576** |


- Kernel start skew:                 1.020 ms
- Longest local kernel:              5.126 ms
- Earliest start to latest finish:   5.394 ms
- Corrected payload lower bound:     3.089 ms

So the CP8EP8 decomposes as 1.566ms balanced ideal x 1.97 routing hotspot x 1.755 NCCL/transport efficiency = 5.39ms.  
This seems quite reasonable. 

The 1.755 isn't just due to NCCL being slow, its mainly due to the skew in start/end times. For example, rank 2, which receives 247832-30946=216886 tokens; this only starts at t=1.02. Also its due to expert routing imbalance.

So I think intra-node comm looks reasonable.

### Simulation
Ran the exact A2A with those routing numbers in the table above, on 8 GPUs. 

Launch skew, 0.005-0.010ms.

| Routing                | Wire ideal | Slowest-rank median | Effective payload BW |
|------------------------|------------|---------------------|---------------------|
| Balanced               | 1.566 ms   | 2.376 ms            | 593 GB/s            |
| Exact hotspot (rank 5) | 3.089 ms   | 4.686 ms            | 593 GB/s            |
| Exact collective tail  |      —     | 5.171 ms            |         —           |

So it seems like the NVLink is effectively running at 593GB/s, which is 66% of 900GB/s. This is realistic because we have to account for things like protocol overhead and scheduling etc, and the data we are sending here is not that big.  

So it seems like our NVLink is running fine.



# 2. CP16EP16, S = 131072.



## Assumptions

- Assume all GPUs have even routing (the tokens are evenly distributed to experts and thus each GPU receives the same amount of tokens)
- NVLink is at full efficiency; 900GB/s unidirectional.
- RoCE is at full efficiency; 100GB/s unidirectional.

S/CP = 8192 toks/GPU.

On one GPU, we send 8192, and receive 8192.

topk (k=8) routing, so we send 8192 * 8 tok.

half of these stay on the same node, half of them go to the other node.

So send 8192 * 8 * 1/2 toks.

Each tok has h=6144, and is in bf16 (2 bytes)

```text
So 8192 * 8 * 1/2 * 6144 * 2 bytes sent, over RoCE which is 100GB/s unidirectional.
=4.03ms
```

Assume they cross the RoCE rail first, and then travel through NVLink on the destination node.

The GPU on the other side of the RoCE rail now needs to send this data through NVLink, at 900GB/s unidirectional. It keeps 1/8 of it and sends 7/8 of it to other GPUs on the same node, via NVLink.

```text
So add (8192 * 8 * 1/2 * 6144 * 2) * 7/8 bytes at 900GB/s.
= 0.39ms

So total 4.03 + 0.39 = 4.42ms.
```



## REALITY:
We see on rank 0, 25.033ms.  

Firstly, we have to consider the routing imbalance penalty: rank 4 is the cross-node receive hotspot, our routing matrix shows that 114670 tokens from ranks 8..15 (rank 1) get routed to rank 4, meaning they have to cross RoCE.  

So 114670 * 6144 * 2 @ 100GB/s = 14.1ms.  

As always, we are limited by the slowest rank, so this is the lower bound.  





In [1]:
S = 131_072
hidden = 6_144
bytes_per_element = 2  # bf16
top_k = 8
nvlink_GBps = 900.0
roce_GBps = 100.0

def transfer_ms(tokens, bandwidth_GBps):
    payload_bytes = tokens * hidden * bytes_per_element
    return payload_bytes / (bandwidth_GBps * 1e9) * 1e3

print(f'S={S:,}, hidden={hidden:,}, top_k={top_k}, bf16 bytes={bytes_per_element}')


S=131,072, hidden=6,144, top_k=8, bf16 bytes=2


In [2]:
# CP8/EP8: balanced expectation and rank-0 lower bound from the Markdown.
cp = ep = 8
tokens_per_gpu = S // cp
assignments_per_gpu = tokens_per_gpu * top_k
balanced_remote_tokens = assignments_per_gpu * (ep - 1) // ep
balanced_ideal_ms = transfer_ms(balanced_remote_tokens, nvlink_GBps)

rank0_self = 30_279
rank0_total_sent = 131_072
rank0_total_received = 241_130
rank0_remote_sent = rank0_total_sent - rank0_self
rank0_remote_received = rank0_total_received - rank0_self
rank0_send_bound_ms = transfer_ms(rank0_remote_sent, nvlink_GBps)
rank0_receive_bound_ms = transfer_ms(rank0_remote_received, nvlink_GBps)
rank0_bound_ms = max(rank0_send_bound_ms, rank0_receive_bound_ms)

print('CP8/EP8 balanced and rank-0 calculations')
print(f'tokens/GPU:                 {tokens_per_gpu:,}')
print(f'assignments/GPU:            {assignments_per_gpu:,}')
print(f'balanced remote tokens:     {balanced_remote_tokens:,}')
print(f'balanced ideal:             {balanced_ideal_ms:.6f} ms')
print(f'rank 0 remote sent:         {rank0_remote_sent:,}')
print(f'rank 0 remote received:     {rank0_remote_received:,}')
print(f'rank 0 send bound:          {rank0_send_bound_ms:.6f} ms')
print(f'rank 0 receive bound:       {rank0_receive_bound_ms:.6f} ms')
print(f'rank 0 lower bound:         {rank0_bound_ms:.6f} ms')


CP8/EP8 balanced and rank-0 calculations
tokens/GPU:                 16,384
assignments/GPU:            131,072
balanced remote tokens:     114,688
balanced ideal:             1.565873 ms
rank 0 remote sent:         100,793
rank 0 remote received:     210,851
rank 0 send bound:          1.376160 ms
rank 0 receive bound:       2.878819 ms
rank 0 lower bound:         2.878819 ms


In [3]:
# CP8/EP8 all-rank token-dispatch data.
received = [241_130, 2_772, 247_832, 108_328, 3_042, 258_648, 20_181, 166_643]
self_tokens = [30_281, 371, 30_946, 13_543, 357, 32_431, 2_538, 20_830]
t_start_ms = [0.000000, 0.541533, 1.020281, 0.557426, 0.266277, 0.133229, 0.134093, 0.272354]
t_elapsed_ms = [5.125728, 4.129698, 4.367543, 4.629283, 3.726871, 4.786516, 3.797470, 5.121525]
t_end_ms = [start + elapsed for start, elapsed in zip(t_start_ms, t_elapsed_ms)]
balanced_received = S

print('CP8/EP8 all-rank profile')
print(f"{'rank':>4} {'received':>10} {'balanced x':>11} {'t_start':>10} {'t_end':>10} {'elapsed':>10}")
for rank in range(ep):
    print(
        f'{rank:>4} {received[rank]:>10,} {received[rank] / balanced_received:>11.3f} '
        f'{t_start_ms[rank]:>10.6f} {t_end_ms[rank]:>10.6f} {t_elapsed_ms[rank]:>10.6f}'
    )

remote_received = [total - local for total, local in zip(received, self_tokens)]
hotspot_rank = max(range(ep), key=remote_received.__getitem__)
hotspot_remote_tokens = remote_received[hotspot_rank]
corrected_lower_bound_ms = transfer_ms(hotspot_remote_tokens, nvlink_GBps)
collective_window_ms = max(t_end_ms) - min(t_start_ms)
start_skew_ms = max(t_start_ms) - min(t_start_ms)
longest_local_kernel_ms = max(t_elapsed_ms)
routing_hotspot_multiplier = corrected_lower_bound_ms / balanced_ideal_ms
residual_multiplier = collective_window_ms / corrected_lower_bound_ms

total_assignments = assignments_per_gpu * ep
total_remote_assignments = total_assignments - sum(self_tokens)
total_remote_GB = total_remote_assignments * hidden * bytes_per_element / 1e9
aggregate_payload_TBps = total_remote_GB / collective_window_ms
aggregate_peak_TBps = ep * nvlink_GBps / 1e3

print()
print(f'hotspot rank:                         {hotspot_rank}')
print(f'hotspot remote receive:               {hotspot_remote_tokens:,} tokens')
print(f'kernel start skew:                    {start_skew_ms:.6f} ms')
print(f'longest local kernel:                 {longest_local_kernel_ms:.6f} ms')
print(f'earliest start to latest finish:      {collective_window_ms:.6f} ms')
print(f'corrected payload lower bound:        {corrected_lower_bound_ms:.6f} ms')
print(f'routing-hotspot multiplier:           {routing_hotspot_multiplier:.6f}x')
print(f'window / corrected-bound multiplier:  {residual_multiplier:.6f}x')
print(f'decomposition: {balanced_ideal_ms:.6f} × {routing_hotspot_multiplier:.6f} × {residual_multiplier:.6f} = {collective_window_ms:.6f} ms')
print(f'total remote payload:                 {total_remote_GB:.6f} GB')
print(f'aggregate payload throughput:         {aggregate_payload_TBps:.6f} TB/s')
print(f'aggregate one-way peak:               {aggregate_peak_TBps:.6f} TB/s')
print(f'aggregate peak utilization:           {aggregate_payload_TBps / aggregate_peak_TBps:.2%}')


CP8/EP8 all-rank profile
rank   received  balanced x    t_start      t_end    elapsed
   0    241,130       1.840   0.000000   5.125728   5.125728
   1      2,772       0.021   0.541533   4.671231   4.129698
   2    247,832       1.891   1.020281   5.387824   4.367543
   3    108,328       0.826   0.557426   5.186709   4.629283
   4      3,042       0.023   0.266277   3.993148   3.726871
   5    258,648       1.973   0.133229   4.919745   4.786516
   6     20,181       0.154   0.134093   3.931563   3.797470
   7    166,643       1.271   0.272354   5.393879   5.121525

hotspot rank:                         5
hotspot remote receive:               226,217 tokens
kernel start skew:                    1.020281 ms
longest local kernel:                 5.125728 ms
earliest start to latest finish:      5.393879 ms
corrected payload lower bound:        3.088616 ms
routing-hotspot multiplier:           1.972456x
window / corrected-bound multiplier:  1.746374x
decomposition: 1.565873 × 1.972456 ×

In [4]:
# CP16/EP16: the balanced per-rank expectation from the Markdown.
cp = ep = 16
gpus_per_node = 8
tokens_per_gpu = S // cp
assignments_per_gpu = tokens_per_gpu * top_k
balanced_cross_node_tokens = assignments_per_gpu // 2
balanced_same_node_other_tokens = assignments_per_gpu * 7 // 16
balanced_roce_ms = transfer_ms(balanced_cross_node_tokens, roce_GBps)
balanced_nvlink_ms = transfer_ms(balanced_same_node_other_tokens, nvlink_GBps)
original_sequential_estimate_ms = balanced_roce_ms + balanced_nvlink_ms
balanced_node_shared_ms = gpus_per_node * balanced_roce_ms

print('CP16/EP16 balanced calculations')
print(f'tokens/GPU:                         {tokens_per_gpu:,}')
print(f'assignments/GPU:                    {assignments_per_gpu:,}')
print(f'balanced cross-node tokens/GPU:     {balanced_cross_node_tokens:,}')
print(f'balanced RoCE time/GPU:             {balanced_roce_ms:.6f} ms')
print(f'balanced same-node NVLink time/GPU: {balanced_nvlink_ms:.6f} ms')
print(f'original sequential estimate:       {original_sequential_estimate_ms:.6f} ms')
print(f'balanced node-shared rail bound:    {balanced_node_shared_ms:.6f} ms')


CP16/EP16 balanced calculations
tokens/GPU:                         8,192
assignments/GPU:                    65,536
balanced cross-node tokens/GPU:     32,768
balanced RoCE time/GPU:             4.026532 ms
balanced same-node NVLink time/GPU: 0.391468 ms
original sequential estimate:       4.418000 ms
balanced node-shared rail bound:    32.212255 ms


In [5]:
# CP16/EP16 full routing matrix: matrix[source_rank][destination_rank].
routing = [
    [4914, 10228, 138, 21, 14243, 1160, 6577, 112, 200, 2, 10747, 5462, 1282, 0, 9518, 932],
    [4909, 10229, 156, 41, 14167, 1121, 6661, 122, 212, 2, 10807, 5443, 1257, 0, 9464, 945],
    [4864, 10238, 163, 33, 14264, 1232, 6674, 128, 192, 0, 10717, 5391, 1202, 0, 9542, 896],
    [4860, 10157, 156, 19, 14229, 1189, 6653, 119, 222, 2, 10806, 5421, 1251, 0, 9510, 942],
    [4815, 10206, 155, 27, 14293, 1161, 6662, 117, 191, 1, 10713, 5541, 1267, 0, 9515, 872],
    [4838, 10250, 150, 29, 14270, 1222, 6640, 135, 201, 1, 10826, 5286, 1239, 0, 9539, 910],
    [4850, 10183, 134, 21, 14422, 1143, 6693, 121, 195, 1, 10765, 5322, 1298, 0, 9502, 886],
    [4879, 10129, 150, 26, 14396, 1149, 6580, 148, 182, 1, 10857, 5351, 1253, 0, 9490, 945],
    [4853, 10259, 130, 25, 14411, 1198, 6607, 111, 181, 2, 10725, 5404, 1244, 0, 9513, 873],
    [4878, 10110, 137, 22, 14314, 1225, 6663, 127, 173, 1, 10730, 5381, 1325, 0, 9565, 885],
    [4817, 10230, 165, 24, 14359, 1205, 6721, 115, 177, 0, 10780, 5346, 1273, 0, 9475, 849],
    [4852, 10143, 144, 23, 14369, 1192, 6611, 125, 192, 0, 10821, 5483, 1177, 0, 9473, 931],
    [4942, 10251, 133, 21, 14300, 1149, 6667, 130, 171, 0, 10750, 5332, 1308, 0, 9492, 890],
    [4889, 10268, 140, 37, 14345, 1199, 6640, 104, 154, 2, 10717, 5349, 1231, 0, 9553, 908],
    [4834, 10203, 133, 30, 14396, 1165, 6616, 127, 199, 2, 10755, 5433, 1258, 0, 9506, 879],
    [4827, 10229, 161, 29, 14176, 1161, 6687, 136, 182, 1, 10823, 5363, 1316, 0, 9516, 929],
]

received = [sum(row[destination] for row in routing) for destination in range(ep)]
t_start_ms = [0.502346, 0.902836, 0.130834, 0.047987, 0.924598, 0.925955, 0.509314, 0.919605, 0.873794, 0.260420, 0.863885, 0.087614, 0.879336, 0.869377, 0.000000, 0.461044]
t_elapsed_ms = [24.383376, 24.835396, 24.697785, 24.946743, 34.612323, 24.119477, 24.408312, 24.167952, 34.370189, 34.769862, 34.283991, 35.067736, 34.147797, 34.247815, 35.218825, 34.272046]
t_end_ms = [start + elapsed for start, elapsed in zip(t_start_ms, t_elapsed_ms)]

node0_to_node1_tokens = sum(sum(row[8:]) for row in routing[:8])
node1_to_node0_tokens = sum(sum(row[:8]) for row in routing[8:])
node0_to_node1_GB = node0_to_node1_tokens * hidden * bytes_per_element / 1e9
node1_to_node0_GB = node1_to_node0_tokens * hidden * bytes_per_element / 1e9
actual_node_rail_bound_ms = max(node0_to_node1_GB, node1_to_node0_GB) / roce_GBps * 1e3
collective_window_ms = max(t_end_ms) - min(t_start_ms)

print('CP16/EP16 all-rank profile')
print(f"{'rank':>4} {'received':>10} {'balanced x':>11} {'t_start':>10} {'t_end':>10} {'elapsed':>10}")
for rank in range(ep):
    print(
        f'{rank:>4} {received[rank]:>10,} {received[rank] / assignments_per_gpu:>11.3f} '
        f'{t_start_ms[rank]:>10.6f} {t_end_ms[rank]:>10.6f} {t_elapsed_ms[rank]:>10.6f}'
    )

print()
print(f'node 0 -> node 1:                    {node0_to_node1_tokens:,} tokens, {node0_to_node1_GB:.6f} GB')
print(f'node 1 -> node 0:                    {node1_to_node0_tokens:,} tokens, {node1_to_node0_GB:.6f} GB')
print(f'actual node-shared rail lower bound: {actual_node_rail_bound_ms:.6f} ms')
print(f'kernel start skew:                   {max(t_start_ms) - min(t_start_ms):.6f} ms')
print(f'longest local kernel:                {max(t_elapsed_ms):.6f} ms')
print(f'earliest start to latest finish:     {collective_window_ms:.6f} ms')
print(f'heavy-direction effective rate:      {max(node0_to_node1_GB, node1_to_node0_GB) / collective_window_ms * 1e3:.6f} GB/s')


CP16/EP16 all-rank profile
rank   received  balanced x    t_start      t_end    elapsed
   0     77,821       1.187   0.502346  24.885722  24.383376
   1    163,313       2.492   0.902836  25.738232  24.835396
   2      2,345       0.036   0.130834  24.828619  24.697785
   3        428       0.007   0.047987  24.994730  24.946743
   4    228,954       3.494   0.924598  35.536921  34.612323
   5     18,871       0.288   0.925955  25.045432  24.119477
   6    106,352       1.623   0.509314  24.917626  24.408312
   7      1,977       0.030   0.919605  25.087557  24.167952
   8      3,024       0.046   0.873794  35.243983  34.370189
   9         18       0.000   0.260420  35.030282  34.769862
  10    172,339       2.630   0.863885  35.147876  34.283991
  11     86,308       1.317   0.087614  35.155350  35.067736
  12     20,181       0.308   0.879336  35.027133  34.147797
  13          0       0.000   0.869377  35.117192  34.247815
  14    152,173       2.322   0.000000  35.218825  35.2188